# Introduction to Arabic Speech Technologies
## Chapter 1 notebook: Word error rate and character error rate for Arabic

Electronic supplementary material for *Introduction to Arabic Speech Technologies* by Hend S. Al-Khalifa.

Chapter 1 works a word error rate by hand and insists that a reported score is only interpretable alongside the evaluation split and the text normalization that produced it. This notebook implements both metrics, prints the alignment behind them, shows how much the number moves when Arabic normalization choices change, and adds a per-dialect breakdown and a simple significance test.

**Contents**

1. Edit distance, with the alignment printed
2. Word error rate: S, D, I and N
3. Character error rate
4. Arabic normalization switches and what each one costs
5. A per-dialect breakdown
6. Is the gap real? A bootstrap test

**Running it.** The notebook needs only `numpy`, `scipy` and `matplotlib` (see `requirements.txt`). It runs top to bottom with no downloads and no audio files: where a recording is useful, the notebook synthesises one, and a cell is provided for reading your own WAV file instead. Optional cells that need extra packages or internet access are marked *Optional*.


## 1. Edit distance, with the alignment printed

Word error rate is the minimum number of substitutions, deletions and insertions that turn the recognised sequence into the reference, divided by the number of reference words. The table produced below is the alignment that the count comes from.

In [1]:
import numpy as np
import unicodedata, re

def align(ref, hyp):
    """Levenshtein alignment. Returns (ops, S, D, I) with ops as (tag, ref_token, hyp_token)."""
    n, m = len(ref), len(hyp)
    d = np.zeros((n + 1, m + 1), dtype=int)
    d[:, 0] = np.arange(n + 1)
    d[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i, j] = min(d[i - 1, j] + 1,        # deletion
                          d[i, j - 1] + 1,        # insertion
                          d[i - 1, j - 1] + cost) # match or substitution
    i, j, ops = n, m, []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i, j] == d[i - 1, j - 1] + (0 if ref[i - 1] == hyp[j - 1] else 1):
            ops.append(("=" if ref[i - 1] == hyp[j - 1] else "S", ref[i - 1], hyp[j - 1])); i, j = i - 1, j - 1
        elif i > 0 and d[i, j] == d[i - 1, j] + 1:
            ops.append(("D", ref[i - 1], "")); i -= 1
        else:
            ops.append(("I", "", hyp[j - 1])); j -= 1
    ops.reverse()
    S = sum(1 for o in ops if o[0] == "S"); D = sum(1 for o in ops if o[0] == "D"); I = sum(1 for o in ops if o[0] == "I")
    return ops, S, D, I

def show_alignment(ops):
    print(f"{'op':3} {'reference':>18} {'hypothesis':>18}")
    print("-" * 42)
    for tag, r, h in ops:
        print(f"{tag:3} {r:>18} {h:>18}")

In [2]:
reference  = "ذهب الأولاد إلى المدرسة"
hypothesis = "ذهب الاولاد الى المدرسه"

ops, S, D, I = align(reference.split(), hypothesis.split())
show_alignment(ops)
N = len(reference.split())
print(f"\nS={S}  D={D}  I={I}  N={N}   WER = (S+D+I)/N = {(S+D+I)/N:.3f}")

op           reference         hypothesis
------------------------------------------
=                  ذهب                ذهب
S              الأولاد            الاولاد
S                  إلى                الى
S              المدرسة            المدرسه

S=3  D=0  I=0  N=4   WER = (S+D+I)/N = 0.750


Three of the four words differ only in spelling: bare alif for hamzated alif, alif maqṣūra for yāʾ, and hāʾ for tāʾ marbūṭa. Without normalization the score treats them as errors.

## 2. Word error rate and character error rate

Character error rate applies the same alignment to characters. For Arabic it is often the more informative of the two, because a single clitic or spelling difference can turn a whole word into an error.

In [3]:
def wer(ref, hyp):
    ops, S, D, I = align(ref.split(), hyp.split())
    n = max(len(ref.split()), 1)
    return (S + D + I) / n, dict(S=S, D=D, I=I, N=n)

def cer(ref, hyp):
    r, h = list(ref.replace(" ", "")), list(hyp.replace(" ", ""))
    ops, S, D, I = align(r, h)
    n = max(len(r), 1)
    return (S + D + I) / n, dict(S=S, D=D, I=I, N=n)

w, wc = wer(reference, hypothesis)
c, cc = cer(reference, hypothesis)
print(f"WER = {w:.3f}  {wc}")
print(f"CER = {c:.3f}  {cc}")

WER = 0.750  {'S': 3, 'D': 0, 'I': 0, 'N': 4}
CER = 0.150  {'S': 3, 'D': 0, 'I': 0, 'N': 20}


## 3. Arabic normalization switches

Each switch below is a scoring convention, not a claim that two forms mean the same thing. Report which ones were applied; a result without them is not comparable with another.

In [4]:
DIACRITICS = re.compile(r"[\u064B-\u0652\u0670\u0640]")   # harakat, dagger alif, tatweel

def normalize(text, strip_diacritics=True, unify_alif=True, ta_marbuta=True,
              alif_maqsura=True, strip_punct=True, digits=True):
    t = unicodedata.normalize("NFC", text)
    if strip_diacritics: t = DIACRITICS.sub("", t)
    if unify_alif:       t = re.sub("[\u0622\u0623\u0625]", "\u0627", t)   # آ أ إ -> ا
    if ta_marbuta:       t = t.replace("\u0629", "\u0647")                  # ة -> ه
    if alif_maqsura:     t = t.replace("\u0649", "\u064A")                  # ى -> ي
    if strip_punct:      t = re.sub(r"[^\w\s\u0600-\u06FF]", " ", t)
    if digits:           t = t.translate(str.maketrans("\u0660\u0661\u0662\u0663\u0664\u0665\u0666\u0667\u0668\u0669", "0123456789"))
    return re.sub(r"\s+", " ", t).strip()

settings = [
    ("no normalization", dict(strip_diacritics=False, unify_alif=False, ta_marbuta=False, alif_maqsura=False)),
    ("diacritics only", dict(strip_diacritics=True, unify_alif=False, ta_marbuta=False, alif_maqsura=False)),
    ("+ alif forms", dict(strip_diacritics=True, unify_alif=True, ta_marbuta=False, alif_maqsura=False)),
    ("+ ta marbuta", dict(strip_diacritics=True, unify_alif=True, ta_marbuta=True, alif_maqsura=False)),
    ("+ alif maqsura (full)", dict(strip_diacritics=True, unify_alif=True, ta_marbuta=True, alif_maqsura=True)),
]
print(f"{'setting':24} {'WER':>6} {'CER':>6}")
print("-" * 38)
for name, kw in settings:
    r, h = normalize(reference, **kw), normalize(hypothesis, **kw)
    print(f"{name:24} {wer(r,h)[0]:6.3f} {cer(r,h)[0]:6.3f}")

setting                     WER    CER
--------------------------------------
no normalization          0.750  0.150
diacritics only           0.750  0.150
+ alif forms              0.250  0.050
+ ta marbuta              0.000  0.000
+ alif maqsura (full)     0.000  0.000


The same pair of transcripts can score anywhere between a large error rate and zero, depending only on the normalization applied. This is the point Chapter 1 makes and Chapter 4 turns into an engineering checklist.

## 4. A per-dialect breakdown

An overall average can hide a group the system serves badly. The toy evaluation set below carries a dialect label on every utterance, which is all that is needed to report per-group scores beside the overall one.

In [5]:
evalset = [
    # (dialect, reference, hypothesis)
    ("MSA",      "ذهب الأولاد إلى المدرسة",        "ذهب الأولاد إلى المدرسة"),
    ("MSA",      "افتتح المؤتمر في الرياض",        "افتتح المؤتمر في الرياض"),
    ("Gulf",     "وين أقرب محطة بنزين",            "وين أقرب محطة بترول"),
    ("Gulf",     "شلونك اليوم يا صديقي",           "شلونك اليوم يا صديقي"),
    ("Egyptian", "عايز أروح المحطة دلوقتي",        "عايز أروح المحطة دلوقت"),
    ("Maghrebi", "بغيت نمشي للمحطة دابا",          "بغيت نمشي المحطة دبا"),
    ("Maghrebi", "شحال من ساعة باقي",              "شحال من ساعة باقية"),
]

def corpus_wer(pairs, **kw):
    S = D = I = N = 0
    for ref, hyp in pairs:
        r, h = normalize(ref, **kw), normalize(hyp, **kw)
        _, counts = wer(r, h)
        S += counts["S"]; D += counts["D"]; I += counts["I"]; N += counts["N"]
    return (S + D + I) / N, N

full = dict(strip_diacritics=True, unify_alif=True, ta_marbuta=True, alif_maqsura=True)
overall, n_all = corpus_wer([(r, h) for _, r, h in evalset], **full)
print(f"{'group':10} {'WER':>7} {'ref words':>10}")
print("-" * 30)
for dialect in ["MSA", "Gulf", "Egyptian", "Maghrebi"]:
    pairs = [(r, h) for d, r, h in evalset if d == dialect]
    score, n = corpus_wer(pairs, **full)
    print(f"{dialect:10} {score:7.3f} {n:10d}")
print("-" * 30)
print(f"{'overall':10} {overall:7.3f} {n_all:10d}")

group          WER  ref words
------------------------------
MSA          0.000          8
Gulf         0.125          8
Egyptian     0.250          4
Maghrebi     0.375          8
------------------------------
overall      0.179         28


## 5. Is the gap real?

On a small test set a difference of a point or two may not survive resampling. The bootstrap below resamples utterances with replacement and reports how often system B beats system A.

In [6]:
rng = np.random.default_rng(7)

def bootstrap_compare(pairs_a, pairs_b, n_boot=2000, **kw):
    idx = np.arange(len(pairs_a))
    wins = 0
    for _ in range(n_boot):
        sample = rng.choice(idx, size=len(idx), replace=True)
        a, _ = corpus_wer([pairs_a[i] for i in sample], **kw)
        b, _ = corpus_wer([pairs_b[i] for i in sample], **kw)
        wins += (b < a)
    return wins / n_boot

system_a = [(r, h) for _, r, h in evalset]
# system B fixes one Maghrebi word and breaks one Gulf word
system_b = list(system_a)
system_b[5] = ("بغيت نمشي للمحطة دابا", "بغيت نمشي للمحطة دابا")
system_b[2] = ("وين أقرب محطة بنزين", "وين أقرب محطات بترول")

a, _ = corpus_wer(system_a, **full); b, _ = corpus_wer(system_b, **full)
p = bootstrap_compare(system_a, system_b, **full)
print(f"system A WER {a:.3f}   system B WER {b:.3f}")
print(f"system B is better in {p*100:.1f}% of bootstrap resamples")
print("\nWith seven utterances this is a demonstration of the procedure, not evidence about either system.")

system A WER 0.179   system B WER 0.143
system B is better in 54.4% of bootstrap resamples

With seven utterances this is a demonstration of the procedure, not evidence about either system.


## What to report with a score

The dataset and release version, the exact split, the normalization switches used above, whether diacritics were scored, the tokenization, and the per-group results beside the overall one.